In [3]:
import os
import geopandas as gpd
import pandas as pd
import shapely
import fiona
import pyproj
from shapely.geometry import Point
from tqdm import tqdm
# -------------------------------
# 0. Print Version Details
# -------------------------------
print("Version Details:")
print(f"  - geopandas: {gpd.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - shapely: {shapely.__version__}")
print(f"  - fiona: {fiona.__version__}")
print(f"  - pyproj: {pyproj.__version__}")
print("-" * 50)

# Set the base directory for datasets
base_dir = r"F:\EYDS\Dataset\Base_Dataset"

Version Details:
  - geopandas: 1.0.1
  - pandas: 2.2.3
  - shapely: 2.0.7
  - fiona: 1.10.1
  - pyproj: 3.6.1
--------------------------------------------------


In [4]:

# -------------------------------
# 1. Read Building Footprint KML
# -------------------------------
print("Step 1: Loading Building Footprint KML...")

# Load the building footprints from the KML file.
building_kml = os.path.join(base_dir, "Building_Footprint.kml")

try:
    buildings = gpd.read_file(building_kml, driver="KML")
    print(f"  ✅ Loaded {len(buildings)} building footprints.")
except Exception as e:
    print(f"  ❌ Error loading KML: {e}")
    exit()

# Project data to UTM Zone 18N for accurate spatial operations
try:
    buildings_proj = buildings.to_crs(epsg=32618)
    print("  ✅ Successfully projected to EPSG:32618 (meters).")
except Exception as e:
    print(f"  ❌ Projection Error: {e}")
    exit()

print("-" * 50)


Step 1: Loading Building Footprint KML...
  ✅ Loaded 9436 building footprints.
  ✅ Successfully projected to EPSG:32618 (meters).
--------------------------------------------------


In [5]:

# -------------------------------
# 2. Helper Function: Count Buildings
# -------------------------------
def count_buildings_for_points(df, thresholds, source_crs="EPSG:4326", target_crs="EPSG:32618"):
    """
    For each point in df (which must contain 'Latitude' and 'Longitude'),
    count the number of building footprints (from buildings_proj) within various distance thresholds.
    
    Returns a DataFrame with the original data plus new columns for each threshold.
    """
    print(f"  ▶ Processing {len(df)} points...")

    # Create a GeoDataFrame from the input DataFrame
    try:
        gdf = gpd.GeoDataFrame(
            df.copy(),
            geometry=[Point(xy) for xy in zip(df.Longitude, df.Latitude)],
            crs=source_crs
        )
        print("  ✅ Created GeoDataFrame for input points.")
    except Exception as e:
        print(f"  ❌ Error creating GeoDataFrame: {e}")
        exit()

    # Project points to the same CRS as the buildings
    try:
        gdf = gdf.to_crs(target_crs)
        print("  ✅ Reprojected points to EPSG:32618.")
    except Exception as e:
        print(f"  ❌ Projection Error: {e}")
        exit()

    # Initialize a dictionary for results
    results = df.copy()

    # Process each threshold
    for d in tqdm(thresholds, desc="Processing thresholds"):
        col_name = f"building_count_{d}m"
        counts = []
        print(f"  ▶ Counting buildings within {d}m...")

        # Create buffers and count buildings
        buffers = gdf.geometry.buffer(d)
        for buf in tqdm(buffers, desc=f"    Counting {d}m", leave=False):
            count = buildings_proj[buildings_proj.intersects(buf)].shape[0]
            counts.append(count)

        results[col_name] = counts
        print(f"  ✅ Completed count for {d}m.")

    return results

print("-" * 50)


--------------------------------------------------


In [6]:

# -------------------------------
# 3. Define Distance Thresholds
# -------------------------------
thresholds = [10, 20, 50, 100, 150, 200, 300, 400, 500, 750, 1000]
print(f"Step 3: Using thresholds: {thresholds}")
print("-" * 50)


Step 3: Using thresholds: [10, 20, 50, 100, 150, 200, 300, 400, 500, 750, 1000]
--------------------------------------------------


In [7]:

# -------------------------------
# 4. Process Training and Validation Points
# -------------------------------
print("Step 4: Processing Training and Validation Data...")

# Load training dataset
train_file = os.path.join(base_dir, "Training_data.csv")
try:
    train_df = pd.read_csv(train_file)
    print(f"  ✅ Loaded Training Data: {train_df.shape}")
except Exception as e:
    print(f"  ❌ Error loading training data: {e}")
    exit()

# Load validation/submission dataset
submission_file = os.path.join(base_dir, "Validation_data.csv")
try:
    submission_df = pd.read_csv(submission_file)
    print(f"  ✅ Loaded Validation Data: {submission_df.shape}")
except Exception as e:
    print(f"  ❌ Error loading validation data: {e}")
    exit()

# Compute building counts
print("Processing training data...")
train_with_building_counts = count_buildings_for_points(train_df, thresholds)

print("Processing validation/submission data...")
validation_with_building_counts = count_buildings_for_points(submission_df, thresholds)

print("-" * 50)


Step 4: Processing Training and Validation Data...
  ✅ Loaded Training Data: (500, 323)
  ✅ Loaded Validation Data: (1040, 330)
Processing training data...
  ▶ Processing 500 points...
  ✅ Created GeoDataFrame for input points.
  ✅ Reprojected points to EPSG:32618.


Processing thresholds:   0%|          | 0/11 [00:00<?, ?it/s]

  ▶ Counting buildings within 10m...


Processing thresholds:   9%|▉         | 1/11 [00:00<00:03,  3.19it/s]

  ✅ Completed count for 10m.
  ▶ Counting buildings within 20m...


...

  ✅ Completed count for 500m.
  ▶ Counting buildings within 750m...


Processing thresholds:  91%|█████████ | 10/11 [00:18<00:03,  3.28s/it]

  ✅ Completed count for 750m.
  ▶ Counting buildings within 1000m...


Processing thresholds: 100%|██████████| 11/11 [00:27<00:00,  2.54s/it]

  ✅ Completed count for 1000m.
--------------------------------------------------


In [8]:

# -------------------------------
# 5. Save the Results to CSV Files
# -------------------------------
train_output = os.path.join(base_dir, "Building_count_training.csv")
validation_output = os.path.join(base_dir, "Building_count_validation.csv")

try:
    train_with_building_counts.to_csv(train_output, index=False)
    print(f"  ✅ Training Data Saved: {train_output}")
except Exception as e:
    print(f"  ❌ Error saving training data: {e}")

try:
    validation_with_building_counts.to_csv(validation_output, index=False)
    print(f"  ✅ Validation Data Saved: {validation_output}")
except Exception as e:
    print(f"  ❌ Error saving validation data: {e}")

print("-" * 50)
print("✅ All steps completed successfully!")


  ✅ Training Data Saved: F:\EYDS\Dataset\Base_Dataset\Building_count_training.csv
  ✅ Validation Data Saved: F:\EYDS\Dataset\Base_Dataset\Building_count_validation.csv
--------------------------------------------------
✅ All steps completed successfully!
